# Integración de librerías .dll

En Python carga la DLL mediante `ctypes`, vincula los tipos de parámetros (`c_double`, `c_void_p`, `c_int`) y ofrece una interfaz interactiva.

In [1]:
import ctypes
from pathlib import Path

In [2]:
# Carga de la librería .dll generada con C++ (en Windows)
dll_path = Path.cwd() / "engine" / "figuras_engine.dll"
if not dll_path.is_file():
    raise FileNotFoundError(f"No se encontró la DLL: {dll_path.resolve()}")

try:
    figuras_engine = ctypes.CDLL(str(dll_path.resolve()))
except OSError as exc:
    if getattr(exc, "winerror", None) == 1114:
        raise RuntimeError(
            "Windows no pudo inicializar la DLL. Recompílala desde engine_api.cpp "
            "y Figura.h con TDM-GCC x64; no se puede corregir este error desde ctypes."
        ) from exc
    raise

In [8]:
# Mapear firmas de funciones exportadas por la DLL
# Por ejemplo, si la DLL tiene una función llamada 'crear_figura' 
# que toma un int y devuelve un puntero a una estructura, se puede definir así:
# figuras_engine.crear_figura.argtypes = [ctypes.c_int]
# figuras_engine.crear_figura.restype = ctypes.c_void_p  # Devuelve un puntero

engine = figuras_engine  # Alias para mayor claridad

# Mapeo del constructor del círculo
engine.crearCirculo.argtypes = [ctypes.c_double]  # Radio del círculo
engine.crearCirculo.restype = ctypes.c_void_p  # Devuelve un puntero

# Mapeo del constructor del cuadrado
engine.crearCuadro.argtypes = [ctypes.c_double]  # Lado del cuadrado
engine.crearCuadro.restype = ctypes.c_void_p  # Devuelve un puntero

# Mapeo de las funciones para calcular área y perímetro de las figuras

engine.calcularArea.argtypes = [ctypes.c_void_p]  # Puntero a la figura
engine.calcularArea.restype = ctypes.c_double  # Devuelve un double

engine.calcularPerimetro.argtypes = [ctypes.c_void_p]  # Puntero a la figura
engine.calcularPerimetro.restype = ctypes.c_double  # Devuelve un double

engine.destruirFigura.argtypes = [ctypes.c_void_p]  # Puntero a la figura
engine.destruirFigura.restype = None  # No devuelve nada

In [10]:
# Instanciación de la figura círculo (por ejemplo, un círculo con radio 5.0)
radio = 5.0
figura = engine.crearCirculo(radio)
area = engine.calcularArea(figura)
perimetro = engine.calcularPerimetro(figura)  # Suponiendo que también hay una función para calcular el perímetro
print(f"Área del círculo con radio {radio}: {area}")
print(f"Perímetro del círculo con radio {radio}: {perimetro}")
engine.destruirFigura(figura)  # Liberar memoria de la figura

Área del círculo con radio 5.0: 78.54
Perímetro del círculo con radio 5.0: 31.416


In [11]:
# Instanciación de la figura Cuadro (por ejemplo, un cuadrado con lado 4.0)
lado = 4.0
figura = engine.crearCuadro(lado)
area = engine.calcularArea(figura)
perimetro = engine.calcularPerimetro(figura)  
print(f"Área del cuadrado con lado {lado}: {area}")
print(f"Perímetro del cuadrado con lado {lado}: {perimetro}")
engine.destruirFigura(figura)  # Liberar memoria de la figura

Área del cuadrado con lado 4.0: 16.0
Perímetro del cuadrado con lado 4.0: 16.0
